# QC

Check quality of downloaded, trimmed data using fastqc

## FastQC

In [ ]:
# Installing FastQC
!apt-get install fastqc

In [ ]:
# Importing required libraries
import os

# Defining function to run FastQC on a file and save results in QC folder
def run_fastqc(file_path):
    file_name = os.path.basename(file_path)
    qc_folder = 'QC'
    qc_path = os.path.join(qc_folder, file_name.replace('.fastq.gz', '_fastqc.html'))
    !fastqc $file_path --outdir=$qc_folder
    return qc_path

# Creating QC folder if it doesn't exist
qc_folder = 'QC'
if not os.path.exists(qc_folder):
    os.makedirs(qc_folder)

# Running FastQC on all .fastq.gz files in the current directory and subdirectories
for subdir, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.fastq.gz'):
            file_path = os.path.join(subdir, file)
            qc_path = run_fastqc(file_path)
            print(f'FastQC results for {file_path} saved in {qc_path}')


## MultiQC

In [ ]:
# Installing MultiQC
!pip install --upgrade --force-reinstall git+https://github.com/ewels/MultiQC.git

In [ ]:
#change dir to where the trimmed files are stored
%cd /Fastq/Trimmed

In [ ]:
# Importing required libraries
import os

# Running MultiQC on the FastQC output files in the QC folder
qc_folder = 'QC'
output_folder = qc_folder
!multiqc $qc_folder --outdir=$output_folder

# Printing the path to the MultiQC report
report_path = os.path.join(qc_folder, 'multiqc_report.html')
print(f'MultiQC report saved in {report_path}')


# Download and install Kallisto

In [ ]:
#DOWNLOAD and INSTALL kallisto

!wget https://github.com/pachterlab/kallisto/releases/download/v0.46.2/kallisto_linux-v0.46.2.tar.gz
!tar -xf kallisto_linux-v0.46.2.tar.gz
!cp kallisto/kallisto /usr/local/bin/

In [ ]:
#DOWNLOAD cDNA
!wget https://ftp.ensembl.org/pub/release-109/fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz
!wget https://ftp.ensembl.org/pub/release-109/fasta/homo_sapiens/ncrna/Homo_sapiens.GRCh38.ncrna.fa.gz
!cat Homo_sapiens.GRCh38.cdna.all.fa.gz Homo_sapiens.GRCh38.ncrna.fa.gz > Homo_sapiens.GRCh38.rna.fa.gz

In [ ]:
#make the indexes
!kallisto index --make-unique -i kallisto_index.idx Homo_sapiens.GRCh38.cdna.rna.fa.gz
!cp kallisto_index.idx /Kallisto_indexes/

In [ ]:
!cp /Kallisto_indexes/kallisto_index.idx .

# Alignment Using Kallisto

In [ ]:
# Data has already been trimmed following sequencing as per the methods section
%cd /Fastq/Trimmed

In [ ]:
%%!
# Directory where your script is running
BASE_DIR="/Fastq/Trimmed/"

# Directory where the index file is located
INDEX_DIR="/Kallisto_indexes/kallisto_index.idx"

# Directory where the output should be directed
OUTPUT_DIR="/Aligned/"

cd $BASE_DIR
for DIR in $(ls -d */); do
  base=$(echo $DIR|awk '{print substr($0, 1, length()-1)}')
  echo $base
  cd $BASE_DIR$DIR
  files=$(find ./ -type f \( -name "*R1.fastq.gz" -o -name "*R2.fastq.gz" \))
  echo "Running Kallisto with the following code"
  echo "kallisto quant -i "$INDEX_DIR" -o "${OUTPUT_DIR}$DIR" -t 8 --bias -b 50 $files"

  # Check if the output directory already exists
  if [ ! -d "${OUTPUT_DIR}$DIR" ]; then
    kallisto quant -i "$INDEX_DIR" -o "${OUTPUT_DIR}$DIR" -t 4 --bias -b 50 $files | tee -a log
  else
    echo "Output for $DIR already exists. Skipping this iteration." | tee -a log
  fi

  cd ..
done
